# CNN-MCL → BI-LSTM → FWA → RandomForest on NSL-KDD

Two-phase pipeline: (1) train the DL extractor with a temporary softmax head, (2) fit a RandomForest on the extracted 4H features. Paper reference: Hashmi, Barukab & Hamza Osman, PLOS ONE 19(5), 2024.

In [21]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import torch

from nids_dl import RFConfig, TrainConfig, evaluate, extract_features, fit_rf, train_extractor
from nids_dl.data import load_processed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
  print(f"Using {torch.cuda.device_count()} GPU(s):")
  for i in range(torch.cuda.device_count()):
      print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}")
else:
  print("No GPU found, using CPU")


Using 2 GPU(s):
  cuda:0  NVIDIA GeForce RTX 5090
  cuda:1  NVIDIA GeForce RTX 5090


In [17]:
DEVICE

'cuda'

In [6]:
!nvidia-smi

Fri Apr 24 16:25:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.21                 Driver Version: 596.21         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090      WDDM  |   00000000:02:00.0  On |                  N/A |
|  0%   51C    P8             46W /  600W |    3169MiB /  32607MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Load preprocessed NSL-KDD

In [22]:
train = load_processed(ROOT / "data" / "processed" / "train.pt")
test = load_processed(ROOT / "data" / "processed" / "test.pt")

X_tr, y_tr = train["X"], train["y_bin"]
X_te, y_te = test["X"], test["y_bin"]
X_tr.shape, X_te.shape, int(y_tr.max().item()) + 1

(torch.Size([125973, 120]), torch.Size([22544, 120]), 2)

## 2. Phase 1 — train the DL feature extractor

Cross-entropy on the temporary softmax head, Adam, with the MCL prediction-error-filter constraint re-applied after every step.

In [23]:
cfg = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="binary", log_every=0, seed=0)
extractor, history = train_extractor(X_tr, y_tr, cfg, X_val=X_te, y_val=y_te)
history[-1]

{'epoch': 9,
 'train_loss': 0.06957284153781985,
 'train_acc': 0.9752804172322641,
 'val_loss': 0.8166122634077512,
 'val_acc': 0.7551011355571328}

In [7]:
import pandas as pd

pd.DataFrame(history)

,epoch,train_loss,train_acc,val_loss,val_acc
0,0,0.293270,0.874719,0.563809,0.785575
1,1,0.128536,0.956618,0.720008,0.805447
2,2,0.098428,0.968342,0.717435,0.808730
3,3,0.082992,0.972454,0.710266,0.801100
4,4,0.073138,0.975582,0.746713,0.765215
5,5,0.068100,0.977082,0.799670,0.804693
6,6,0.065633,0.977590,0.760503,0.775151
7,7,0.061000,0.979265,0.771683,0.803318
8,8,0.063339,0.978678,0.725712,0.797285
9,9,0.054304,0.981742,0.774513,0.783667


## 3. Extract features and fit RandomForest

In [8]:
Fe_tr = extract_features(extractor, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te = extract_features(extractor, X_te, batch_size=512, device=DEVICE).numpy()
Fe_tr.shape, Fe_te.shape

((125973, 256), (22544, 256))

In [9]:
clf = fit_rf(Fe_tr, y_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_train = evaluate(clf, Fe_tr, y_tr.numpy())
metrics_test = evaluate(clf, Fe_te, y_te.numpy())
{
    "train": {k: metrics_train[k] for k in ("accuracy", "precision", "recall", "f1")},
    "test": {k: metrics_test[k] for k in ("accuracy", "precision", "recall", "f1")},
}

{'train': {'accuracy': 0.999944432537131,
  'precision': 0.9999488307834007,
  'recall': 0.9999317755415317,
  'f1': 0.9999403030897415},
 'test': {'accuracy': 0.7791430092264017,
  'precision': 0.9266623207301173,
  'recall': 0.6646146653159822,
  'f1': 0.774061805145891}}

In [10]:
print(metrics_test["report"])
metrics_test["confusion_matrix"]

              precision    recall  f1-score   support

           0       0.68      0.93      0.78      9711
           1       0.93      0.66      0.77     12833

    accuracy                           0.78     22544
   macro avg       0.80      0.80      0.78     22544
weighted avg       0.82      0.78      0.78     22544



array([[9036,  675],
       [4304, 8529]])

## 4. Multi-class variant (5 classes: Normal/DoS/Probe/R2L/U2R)

In [11]:
ym_tr, ym_te = train["y_mul"], test["y_mul"]
cfg_m = TrainConfig(epochs=10, batch_size=256, lr=1e-3, device=DEVICE, target="multi", seed=0)
extractor_m, history_m = train_extractor(X_tr, ym_tr, cfg_m, X_val=X_te, y_val=ym_te)
Fe_tr_m = extract_features(extractor_m, X_tr, batch_size=512, device=DEVICE).numpy()
Fe_te_m = extract_features(extractor_m, X_te, batch_size=512, device=DEVICE).numpy()
clf_m = fit_rf(Fe_tr_m, ym_tr.numpy(), RFConfig(n_estimators=200, class_weight="balanced", random_state=0))
metrics_te_m = evaluate(clf_m, Fe_te_m, ym_te.numpy())
print(metrics_te_m["report"])
{k: metrics_te_m[k] for k in ("accuracy", "precision", "recall", "f1")}

              precision    recall  f1-score   support

           0       0.65      0.98      0.78      9711
           1       0.97      0.77      0.86      7460
           2       0.83      0.60      0.70      2421
           3       0.96      0.03      0.06      2885
           4       0.00      0.00      0.00        67

    accuracy                           0.75     22544
   macro avg       0.68      0.48      0.48     22544
weighted avg       0.81      0.75      0.70     22544



{'accuracy': 0.7470723917672107,
 'precision': 0.6798153446608264,
 'recall': 0.47748296099652504,
 'f1': 0.4796168070336827}